# RAN Telemetry — Normalized Transformation (10 Silver Tables)

Transforms RAN telemetry JSON into **10 normalized silver tables**.

| # | Table | Granularity | Key change vs. 5-table design |
|---|---|---|---|
| 1 | `site_snapshot` | 1 row / snapshot | No aggregated sensors; `technologies` kept as ARRAY |
| 2 | `environment_sensors` | 1 row / sensor / snapshot | New — splits temp & humidity sensors out of site_snapshot |
| 3 | `cells` | 1 row / cell / snapshot | Unchanged |
| 4 | `antennas` | 1 row / antenna / snapshot | Split out of old `radio_hardware` union |
| 5 | `radio_units` | 1 row / RU / snapshot | Split out of old `radio_hardware` union |
| 6 | `baseband_units` | 1 row / BBU / snapshot | Split out of old `radio_hardware` union |
| 7 | `batteries` | 1 row / battery / snapshot | Split out of old `power_transport` union |
| 8 | `rectifiers` | 1 row / rectifier / snapshot | Split out of old `power_transport` union |
| 9 | `transport_links` | 1 row / link / snapshot | Split out of old `power_transport` union |
| 10 | `alerts` | 1 row / alert / snapshot | `value` split into `alert_value_num` + `alert_value_str` |

In [ ]:
%idle_timeout 60
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2

## 1. Setup

In [ ]:
import boto3
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql.functions import (
    col,
    current_timestamp,
    explode,
    explode_outer,
    input_file_name,
    lit,
    to_timestamp,
    upper,
    when,
)
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
spark.sparkContext.setLogLevel("WARN")

## 2. Paths

In [ ]:
RAW_PREFIX           = "s3://tower-iti-project/raw-data/ran_telemetry/"
OUTPUT_BASE          = "s3://tower-iti-project/silver/ran_telemetry_normalized"
PROCESSED_FILES_PATH = f"{OUTPUT_BASE}/_state/processed_files"

## 3. Schema

In [ ]:
RAN_SCHEMA = StructType([
    StructField("message_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("sequence_number", LongType(), True),
    StructField("ran_metadata", StructType([
        StructField("site_id", StringType(), True),
        StructField("site_name", StringType(), True),
        StructField("location", StructType([
            StructField("latitude", DoubleType(), True),
            StructField("longitude", DoubleType(), True),
        ]), True),
        StructField("region", StringType(), True),
        StructField("vendor", StringType(), True),
        StructField("technology", ArrayType(StringType()), True),
    ]), True),
    StructField("environment", StructType([
        StructField("status", StringType(), True),
        StructField("op_state", StringType(), True),
        StructField("temperature_sensors", ArrayType(StructType([
            StructField("sensor_id", StringType(), True),
            StructField("value_c", DoubleType(), True),
        ])), True),
        StructField("humidity_sensors", ArrayType(StructType([
            StructField("sensor_id", StringType(), True),
            StructField("value_percent", DoubleType(), True),
        ])), True),
        StructField("door_status", StringType(), True),
        StructField("smoke_detected", BooleanType(), True),
    ]), True),
    StructField("antennas", ArrayType(StructType([
        StructField("antenna_id", StringType(), True),
        StructField("sector_id", StringType(), True),
        StructField("status", StringType(), True),
        StructField("op_state", StringType(), True),
        StructField("mimo_layers", IntegerType(), True),
        StructField("azimuth_degree", IntegerType(), True),
        StructField("tilt_degree", DoubleType(), True),
        StructField("rssi_dbm", DoubleType(), True),
        StructField("snr_db", DoubleType(), True),
    ])), True),
    StructField("baseband_units", ArrayType(StructType([
        StructField("bbu_id", StringType(), True),
        StructField("status", StringType(), True),
        StructField("op_state", StringType(), True),
        StructField("active_users", IntegerType(), True),
        StructField("cpu_utilization_percent", DoubleType(), True),
        StructField("memory_utilization_percent", DoubleType(), True),
        StructField("disk_usage_percent", DoubleType(), True),
        StructField("control_plane_latency_ms", DoubleType(), True),
        StructField("user_plane_latency_ms", DoubleType(), True),
        StructField("process_latency_ms", DoubleType(), True),
    ])), True),
    StructField("transport_links", ArrayType(StructType([
        StructField("link_id", StringType(), True),
        StructField("type", StringType(), True),
        StructField("status", StringType(), True),
        StructField("op_state", StringType(), True),
        StructField("throughput_mbps", DoubleType(), True),
        StructField("utilization_percent", DoubleType(), True),
        StructField("latency_ms", DoubleType(), True),
        StructField("jitter_ms", DoubleType(), True),
        StructField("packet_loss_percent", DoubleType(), True),
    ])), True),
    StructField("cells", ArrayType(StructType([
        StructField("cell_id", StringType(), True),
        StructField("sector_id", StringType(), True),
        StructField("technology", StringType(), True),
        StructField("status", StringType(), True),
        StructField("op_state", StringType(), True),
        StructField("bandwidth_mhz", IntegerType(), True),
        StructField("carrier_frequency_mhz", IntegerType(), True),
        StructField("connected_users", IntegerType(), True),
        StructField("active_users", IntegerType(), True),
        StructField("throughput_downlink_mbps", DoubleType(), True),
        StructField("throughput_uplink_mbps", DoubleType(), True),
        StructField("latency_downlink_ms", DoubleType(), True),
        StructField("latency_uplink_ms", DoubleType(), True),
        StructField("prb_utilization_percent", DoubleType(), True),
        StructField("sinr_db", DoubleType(), True),
        StructField("rsrp_dbm", DoubleType(), True),
        StructField("rsrq_db", DoubleType(), True),
        StructField("cqi_avg", DoubleType(), True),
        StructField("handover_success_rate_percent", DoubleType(), True),
        StructField("handover_attempts", IntegerType(), True),
        StructField("handover_failures", IntegerType(), True),
        StructField("rrc_success_rate_percent", DoubleType(), True),
        StructField("rrc_connection_attempts", IntegerType(), True),
        StructField("erab_setup_success_rate_percent", DoubleType(), True),
        StructField("call_drop_rate_percent", DoubleType(), True),
        StructField("abnormal_release_rate_percent", DoubleType(), True),
        StructField("harq_retransmission_rate_percent", DoubleType(), True),
        StructField("bler_uplink_percent", DoubleType(), True),
        StructField("bler_downlink_percent", DoubleType(), True),
        StructField("spectral_efficiency_bps_per_hz", DoubleType(), True),
    ])), True),
    StructField("radio_units", ArrayType(StructType([
        StructField("ru_id", StringType(), True),
        StructField("sector_id", StringType(), True),
        StructField("status", StringType(), True),
        StructField("op_state", StringType(), True),
        StructField("tx_power_watts", DoubleType(), True),
        StructField("rx_signal_strength_dbm", DoubleType(), True),
        StructField("current_ampere", DoubleType(), True),
        StructField("voltage_volt", DoubleType(), True),
        StructField("temperature_c", DoubleType(), True),
        StructField("throughput_mbps", DoubleType(), True),
        StructField("packet_error_rate", DoubleType(), True),
        StructField("vswr", DoubleType(), True),
    ])), True),
    StructField("power_system", StructType([
        StructField("status", StringType(), True),
        StructField("batteries", ArrayType(StructType([
            StructField("battery_id", StringType(), True),
            StructField("status", StringType(), True),
            StructField("op_state", StringType(), True),
            StructField("charge_percent", DoubleType(), True),
            StructField("temperature_c", DoubleType(), True),
        ])), True),
        StructField("rectifiers", ArrayType(StructType([
            StructField("rectifier_id", StringType(), True),
            StructField("status", StringType(), True),
            StructField("op_state", StringType(), True),
            StructField("current_ampere", DoubleType(), True),
            StructField("output_voltage_volt", DoubleType(), True),
        ])), True),
        StructField("generator", StructType([
            StructField("status", StringType(), True),
            StructField("fuel_level_percent", DoubleType(), True),
            StructField("runtime_hours", DoubleType(), True),
        ]), True),
    ]), True),
    StructField("alerts", ArrayType(StructType([
        StructField("alert_id", StringType(), True),
        StructField("severity", StringType(), True),
        StructField("category", StringType(), True),
        StructField("component_type", StringType(), True),
        StructField("component_id", StringType(), True),
        StructField("code", StringType(), True),
        StructField("message", StringType(), True),
        StructField("value", StringType(), True),
    ])), True),
    StructField("alert_summary", StructType([
        StructField("total", IntegerType(), True),
        StructField("critical", IntegerType(), True),
        StructField("warning", IntegerType(), True),
        StructField("info", IntegerType(), True),
        StructField("highest_severity", StringType(), True),
    ]), True),
])

## 4. Incremental File Detection

Lists all raw JSON files under `RAW_PREFIX`, subtracts files already recorded in the
processed-files manifest, and returns only the new ones for this run.

In [ ]:
def parse_s3_uri(uri):
    bucket_and_key = uri[5:]
    bucket, _, key = bucket_and_key.partition("/")
    return bucket, key

def list_s3_json_files(prefix_uri):
    bucket, prefix = parse_s3_uri(prefix_uri)
    client = boto3.client("s3")
    paginator = client.get_paginator("list_objects_v2")
    files = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".json"):
                files.append(f"s3://{bucket}/{obj['Key']}")
    return sorted(files)

def read_processed_files(path):
    try:
        return {
            row.source_file
            for row in spark.read.parquet(path).select("source_file").distinct().collect()
        }
    except Exception:
        return set()

all_files       = list_s3_json_files(RAW_PREFIX)
processed_files = read_processed_files(PROCESSED_FILES_PATH)
new_files       = [f for f in all_files if f not in processed_files]

print(f"Raw files found      : {len(all_files)}")
print(f"Already processed    : {len(processed_files)}")
print(f"New files to process : {len(new_files)}")

## 5. Load Raw Data

In [ ]:
if new_files:
    df_raw = (
        spark.read
        .schema(RAN_SCHEMA)
        .json(new_files)
        .withColumn("source_file", input_file_name())
    )
else:
    df_raw = (
        spark.createDataFrame([], RAN_SCHEMA)
        .withColumn("source_file", input_file_name())
    )

print(f"New snapshots loaded : {df_raw.count()}")
df_raw.printSchema()

## 6. Shared Base Columns

`BASE_COLS` is used in the first `select` (raw nested paths).  
`BASE_PASSTHROUGH` is used in the second `select` after `explode` (flat aliases).  
Every silver table carries these 7 columns so all tables are joinable by snapshot.

In [ ]:
BASE_COLS = [
    col("message_id"),
    col("source_file"),
    to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("snapshot_time"),
    col("sequence_number"),
    col("ran_metadata.site_id").alias("site_id"),
    col("ran_metadata.site_name").alias("site_name"),
    col("ran_metadata.region").alias("region"),
]

BASE_PASSTHROUGH = [
    col("message_id"),
    col("source_file"),
    col("snapshot_time"),
    col("sequence_number"),
    col("site_id"),
    col("site_name"),
    col("region"),
]

---
## 7. Silver Table Definitions

### 7a. site_snapshot

One wide row per snapshot. Keeps site metadata, environment status, generator state,
and alert summary counts. Sensor readings move to `environment_sensors`.
`technologies` is kept as `ARRAY<STRING>` instead of a comma-joined string.

In [ ]:
df_site_snapshot = df_raw.select(
    *BASE_COLS,
    col("ran_metadata.vendor").alias("vendor"),
    col("ran_metadata.technology").alias("technologies"),
    col("ran_metadata.location.latitude").alias("latitude"),
    col("ran_metadata.location.longitude").alias("longitude"),
    col("environment.status").alias("env_status"),
    col("environment.op_state").alias("env_op_state"),
    col("environment.smoke_detected"),
    col("environment.door_status"),
    col("power_system.generator.status").alias("gen_status"),
    col("power_system.generator.fuel_level_percent").alias("fuel_level_pct"),
    col("power_system.generator.runtime_hours"),
    col("alert_summary.total").alias("total_alerts"),
    col("alert_summary.critical").alias("critical_alerts"),
    col("alert_summary.warning").alias("warning_alerts"),
    col("alert_summary.info").alias("info_alerts"),
    col("alert_summary.highest_severity"),
)

df_site_snapshot.show(5, truncate=False)

### 7b. environment_sensors

One row per sensor per snapshot. Temperature and humidity sensors are unioned into
one table with a `sensor_type` discriminator (`TEMPERATURE` / `HUMIDITY`).
`status` is derived from thresholds: temperature >40°C → CRITICAL, >37°C → HIGH;
humidity >80% → HIGH.

In [ ]:
df_temp_sensors = df_raw.select(
    *BASE_COLS,
    explode(col("environment.temperature_sensors")).alias("sensor"),
).select(
    *BASE_PASSTHROUGH,
    lit("TEMPERATURE").alias("sensor_type"),
    col("sensor.sensor_id"),
    col("sensor.value_c").alias("value"),
    lit("C").alias("unit"),
    when(col("sensor.value_c") > 40, "CRITICAL")
    .when(col("sensor.value_c") > 37, "HIGH")
    .otherwise("OK").alias("status"),
)

df_hum_sensors = df_raw.select(
    *BASE_COLS,
    explode(col("environment.humidity_sensors")).alias("sensor"),
).select(
    *BASE_PASSTHROUGH,
    lit("HUMIDITY").alias("sensor_type"),
    col("sensor.sensor_id"),
    col("sensor.value_percent").alias("value"),
    lit("percent").alias("unit"),
    when(col("sensor.value_percent") > 80, "HIGH")
    .otherwise("OK").alias("status"),
)

df_environment_sensors = df_temp_sensors.unionByName(df_hum_sensors)

df_environment_sensors.show(10, truncate=False)

### 7c. cells

One row per cell per snapshot. All radio KPIs: capacity, coverage, quality,
mobility, and reliability.

In [ ]:
df_cells = df_raw.select(
    *BASE_COLS,
    explode(col("cells")).alias("cell"),
).select(
    *BASE_PASSTHROUGH,
    col("cell.cell_id"),
    col("cell.sector_id"),
    col("cell.technology"),
    col("cell.status").alias("cell_status"),
    col("cell.op_state").alias("cell_op_state"),
    col("cell.bandwidth_mhz"),
    col("cell.carrier_frequency_mhz"),
    col("cell.connected_users"),
    col("cell.active_users"),
    col("cell.throughput_downlink_mbps"),
    col("cell.throughput_uplink_mbps"),
    col("cell.latency_downlink_ms"),
    col("cell.latency_uplink_ms"),
    col("cell.prb_utilization_percent"),
    col("cell.sinr_db"),
    col("cell.rsrp_dbm"),
    col("cell.rsrq_db"),
    col("cell.cqi_avg"),
    col("cell.handover_success_rate_percent"),
    col("cell.handover_attempts"),
    col("cell.handover_failures"),
    col("cell.rrc_success_rate_percent"),
    col("cell.rrc_connection_attempts"),
    col("cell.erab_setup_success_rate_percent"),
    col("cell.call_drop_rate_percent"),
    col("cell.abnormal_release_rate_percent"),
    col("cell.harq_retransmission_rate_percent"),
    col("cell.bler_uplink_percent"),
    col("cell.bler_downlink_percent"),
    col("cell.spectral_efficiency_bps_per_hz"),
)

df_cells.show(5, truncate=False)

### 7d. antennas

One row per antenna per snapshot. Antenna RF and physical configuration.

In [ ]:
df_antennas = df_raw.select(
    *BASE_COLS,
    explode(col("antennas")).alias("ant"),
).select(
    *BASE_PASSTHROUGH,
    col("ant.antenna_id"),
    col("ant.sector_id"),
    col("ant.status"),
    col("ant.op_state"),
    col("ant.mimo_layers"),
    col("ant.azimuth_degree"),
    col("ant.tilt_degree"),
    col("ant.rssi_dbm"),
    col("ant.snr_db"),
)

df_antennas.show(5, truncate=False)

### 7e. radio_units

One row per radio unit per snapshot. RF power, signal, temperature, and VSWR.
`rx_signal_strength_dbm` is aliased to `rx_signal_dbm` per the schema spec.

In [ ]:
df_radio_units = df_raw.select(
    *BASE_COLS,
    explode(col("radio_units")).alias("ru"),
).select(
    *BASE_PASSTHROUGH,
    col("ru.ru_id"),
    col("ru.sector_id"),
    col("ru.status"),
    col("ru.op_state"),
    col("ru.tx_power_watts"),
    col("ru.rx_signal_strength_dbm").alias("rx_signal_dbm"),
    col("ru.voltage_volt"),
    col("ru.temperature_c"),
    col("ru.throughput_mbps"),
    col("ru.packet_error_rate"),
    col("ru.vswr"),
)

df_radio_units.show(5, truncate=False)

### 7f. baseband_units

One row per BBU per snapshot. Compute utilization and latency metrics.

In [ ]:
df_baseband_units = df_raw.select(
    *BASE_COLS,
    explode(col("baseband_units")).alias("bbu"),
).select(
    *BASE_PASSTHROUGH,
    col("bbu.bbu_id"),
    col("bbu.status"),
    col("bbu.op_state"),
    col("bbu.active_users"),
    col("bbu.cpu_utilization_percent").alias("cpu_pct"),
    col("bbu.memory_utilization_percent").alias("memory_pct"),
    col("bbu.disk_usage_percent").alias("disk_pct"),
    col("bbu.control_plane_latency_ms").alias("control_latency_ms"),
    col("bbu.user_plane_latency_ms").alias("user_latency_ms"),
    col("bbu.process_latency_ms"),
)

df_baseband_units.show(5, truncate=False)

### 7g. batteries

One row per battery per snapshot. Charge percentage, temperature, and health.
Note: raw battery objects do not include a `voltage_volt` field.

In [ ]:
df_batteries = df_raw.select(
    *BASE_COLS,
    explode(col("power_system.batteries")).alias("bat"),
).select(
    *BASE_PASSTHROUGH,
    col("bat.battery_id"),
    col("bat.status"),
    col("bat.op_state"),
    col("bat.charge_percent").alias("charge_pct"),
    col("bat.temperature_c"),
)

df_batteries.show(5, truncate=False)

### 7h. rectifiers

One row per rectifier per snapshot. Current, voltage, and health.

In [ ]:
df_rectifiers = df_raw.select(
    *BASE_COLS,
    explode(col("power_system.rectifiers")).alias("rec"),
).select(
    *BASE_PASSTHROUGH,
    col("rec.rectifier_id"),
    col("rec.status"),
    col("rec.op_state"),
    col("rec.current_ampere"),
    col("rec.output_voltage_volt"),
)

df_rectifiers.show(5, truncate=False)

### 7i. transport_links

One row per backhaul link per snapshot. Throughput, utilization, latency, jitter,
and packet loss. `link_type` is uppercased to normalise raw values (`fiber` → `FIBER`).

In [ ]:
df_transport_links = df_raw.select(
    *BASE_COLS,
    explode(col("transport_links")).alias("link"),
).select(
    *BASE_PASSTHROUGH,
    col("link.link_id"),
    upper(col("link.type")).alias("link_type"),
    col("link.status"),
    col("link.op_state"),
    col("link.throughput_mbps"),
    col("link.utilization_percent"),
    col("link.latency_ms"),
    col("link.jitter_ms"),
    col("link.packet_loss_percent"),
)

df_transport_links.show(5, truncate=False)

### 7j. alerts

One row per alert per snapshot. `explode_outer` preserves healthy snapshots
(alert columns are NULL when there are no alerts).

Raw `alert.value` contains mixed types (`"68.4"` or `"DOWN"`). It is split into:
- `alert_value_num` — DOUBLE when the value parses as a number, else NULL
- `alert_value_str` — STRING when the value cannot be parsed as a number, else NULL

In [ ]:
df_alerts = df_raw.select(
    *BASE_COLS,
    explode_outer(col("alerts")).alias("alert"),
).select(
    *BASE_PASSTHROUGH,
    col("alert.alert_id"),
    col("alert.severity"),
    col("alert.category"),
    col("alert.component_type"),
    col("alert.component_id"),
    col("alert.code"),
    col("alert.message"),
    col("alert.value").cast(DoubleType()).alias("alert_value_num"),
    when(
        col("alert.value").cast(DoubleType()).isNull() & col("alert.value").isNotNull(),
        col("alert.value"),
    ).cast(StringType()).alias("alert_value_str"),
)

df_alerts.filter(col("alert_id").isNotNull()).show(10, truncate=False)

---
## 8. Write Silver Outputs to S3

In [ ]:
if new_files:
    tables = {
        "site_snapshot":       df_site_snapshot,
        "environment_sensors": df_environment_sensors,
        "cells":               df_cells,
        "antennas":            df_antennas,
        "radio_units":         df_radio_units,
        "baseband_units":      df_baseband_units,
        "batteries":           df_batteries,
        "rectifiers":          df_rectifiers,
        "transport_links":     df_transport_links,
        "alerts":              df_alerts,
    }

    for name, df in tables.items():
        path = f"{OUTPUT_BASE}/{name}"
        df.write.mode("append").partitionBy("region").parquet(path)
        print(f"  written -> {path}")

    (
        spark.createDataFrame([(f,) for f in new_files], ["source_file"])
        .withColumn("processed_at", current_timestamp())
        .write.mode("append").parquet(PROCESSED_FILES_PATH)
    )

    print(f"\nIncremental run complete. Processed {len(new_files)} new raw file(s), 10 normalized silver tables updated.")
else:
    print("No new raw files found. Nothing was written.")

---
## Appendix — Schema Quick Reference

| Table | Rows per snapshot | Primary key | Key columns |
|---|---|---|---|
| `site_snapshot` | 1 | `message_id` | vendor, technologies, env_status, gen_status, fuel_level_pct, highest_severity |
| `environment_sensors` | N (one per sensor) | `message_id + sensor_type + sensor_id` | sensor_type, value, unit, status |
| `cells` | 1 per cell | `message_id + cell_id` | technology, sinr_db, rsrp_dbm, throughput_downlink_mbps, call_drop_rate_percent |
| `antennas` | 1 per antenna | `message_id + antenna_id` | sector_id, mimo_layers, rssi_dbm, snr_db |
| `radio_units` | 1 per RU | `message_id + ru_id` | sector_id, tx_power_watts, rx_signal_dbm, vswr |
| `baseband_units` | 1 per BBU | `message_id + bbu_id` | cpu_pct, memory_pct, control_latency_ms, user_latency_ms |
| `batteries` | 1 per battery | `message_id + battery_id` | charge_pct, temperature_c |
| `rectifiers` | 1 per rectifier | `message_id + rectifier_id` | current_ampere, output_voltage_volt |
| `transport_links` | 1 per link | `message_id + link_id` | link_type, throughput_mbps, utilization_percent, packet_loss_percent |
| `alerts` | 0–N | `message_id + alert_id` | severity, category, component_type, alert_value_num, alert_value_str |